# FM-PCC Colab Workflow

---

## Run Order
1. Mount Drive
2. Clone/Update FM-PCC
3. Boost startup cache wiring
4. Install Miniconda + Create `FMPCC` env
5. Install D3IL
6. Install requirements with pinned compatibility
7. Set runtime env variables
8. Optional W&B relogin
9. Verify dependencies
10. Prepare dataset
11. Smoke test
12. Full train
13. Eval + visualization
14. Archive logs



In [2]:
# @title Notebook Version Marker
from datetime import datetime
import pytz

# Change 'UTC' to your local timezone if preferred (e.g., 'US/Eastern', 'Asia/Shanghai')
TIMEZONE = 'UTC'

now = datetime.now(pytz.timezone(TIMEZONE))
version_mark = now.strftime("%Y.%m.%d_%H%M%S")

print("=" * 40)
print(f"SESSION VERSION: {version_mark}")
print(f"START TIME     : {now.strftime('%A, %b %d, %Y - %I:%M:%S %p %Z')}")
print("=" * 40)
print("Gen3U3")


SESSION VERSION: 2026.04.06_125447
START TIME     : Monday, Apr 06, 2026 - 12:54:47 PM UTC
Gen3U3


## 1) Mount Google Drive



In [3]:
import os
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

FMPCC_ROOT = '/content/drive/MyDrive/FMPCC'
REPO = f'{FMPCC_ROOT}/FM-PCC'
os.makedirs(FMPCC_ROOT, exist_ok=True)

print('Drive mounted')
print('Repo path:', REPO)



Mounted at /content/drive
Drive mounted
Repo path: /content/drive/MyDrive/FMPCC/FM-PCC


## 2) Clone or Update FM-PCC



In [4]:
# @title Git Repository Sync
# @markdown Set this to 1 to replace any edited GitHub files with the latest versions.
# @markdown Your new files/notes will NOT be deleted.
OVERWRITE_LOCAL_CHANGES = "1" # @param [0, 1]
UPDATE_REPO = 1 # @param [0, 1]

import os
os.environ['OVERWRITE_LOCAL_CHANGES'] = str(OVERWRITE_LOCAL_CHANGES)
os.environ['UPDATE_REPO'] = str(UPDATE_REPO)

In [5]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
REPO="$ROOT/FM-PCC"
BRANCH="main" # You can change this if you ever decide to switch branches

mkdir -p "$ROOT"
cd "$ROOT"

if [ ! -d "$REPO/.git" ]; then
  echo "Cloning fresh repository..."
  git clone --recurse-submodules https://github.com/ghubliming/FM-PCC.git
else
  cd "$REPO"

  if [ "$UPDATE_REPO" != "1" ]; then
    echo "Repo update disabled. Using existing local checkout."
  else
    echo "Fetching latest updates from GitHub..."
    git fetch origin "$BRANCH"

    if [ "$OVERWRITE_LOCAL_CHANGES" = "1" ]; then
      echo "OVERWRITE_LOCAL_CHANGES=1: Resetting tracked files to match remote."
      # This command ONLY affects files that exist in the Git repo.
      # It does NOT touch your new notes or results.
      git reset --hard "origin/$BRANCH"
      git submodule update --init --recursive
    else
      # Check if there are any local edits to GitHub files
      CHANGED_FILES="$(git status --porcelain | grep '^ M' || true)"
      if [ -n "$CHANGED_FILES" ]; then
        echo "WARNING: Local edits detected in GitHub files. Skipping update to protect them."
        echo "Set OVERWRITE_LOCAL_CHANGES=1 to replace these files with the remote version."
      else
        echo "No conflicts found. Updating..."
        git merge "origin/$BRANCH"
        git submodule update --init --recursive
      fi
    fi
  fi
fi

echo "------------------------------------------------"
echo "Repo ready: $REPO"
echo "Current Branch: $(git branch --show-current)"
echo "------------------------------------------------"

# This part specifically shows you what is yours (notes/new files)
echo "Your Local Notes & New Files (Not in Git):"
UNTRACKED="$(git ls-files --others --exclude-standard)"
if [ -z "$UNTRACKED" ]; then
  echo "  (None)"
else
  echo "$UNTRACKED"
fi

Fetching latest updates from GitHub...
OVERWRITE_LOCAL_CHANGES=1: Resetting tracked files to match remote.
HEAD is now at a349a19 Merge pull request #14 from ghubliming/update_into_FM (Key Update Gen3U3)
------------------------------------------------
Repo ready: /content/drive/MyDrive/FMPCC/FM-PCC
Current Branch: main
------------------------------------------------
Your Local Notes & New Files (Not in Git):
args_resume_1.json
args_resume_10.json
args_resume_100.json
args_resume_101.json
args_resume_102.json
args_resume_103.json
args_resume_104.json
args_resume_105.json
args_resume_106.json
args_resume_107.json
args_resume_108.json
args_resume_109.json
args_resume_11.json
args_resume_12.json
args_resume_13.json
args_resume_14.json
args_resume_15.json
args_resume_16.json
args_resume_17.json
args_resume_18.json
args_resume_19.json
args_resume_2.json
args_resume_20.json
args_resume_21.json
args_resume_22.json
args_resume_23.json
args_resume_24.json
args_resume_25.json
args_resume_26.jso

From https://github.com/ghubliming/FM-PCC
 * branch            main       -> FETCH_HEAD
   9837996..a349a19  main       -> origin/main
Updating files: 100% (466/466), done.


## 3) Boost Startup Cache Wiring

Keeps the original `/content/miniconda3` path logic, but maps it to Drive so restarts can reuse the same conda env and pip cache.



In [6]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
PERSIST_CONDA="$ROOT/miniconda3"
RUNTIME_CONDA="/content/miniconda3"
PIP_CACHE="$ROOT/.pip-cache"

mkdir -p "$ROOT" "$PIP_CACHE"

if [ -L "$RUNTIME_CONDA" ]; then
  rm -f "$RUNTIME_CONDA"
elif [ -d "$RUNTIME_CONDA" ]; then
  rm -rf "$RUNTIME_CONDA"
fi

ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"

echo "Runtime conda path mapped to: $PERSIST_CONDA"
echo "Persistent pip cache path: $PIP_CACHE"



Runtime conda path mapped to: /content/drive/MyDrive/FMPCC/miniconda3
Persistent pip cache path: /content/drive/MyDrive/FMPCC/.pip-cache


## 4) Install Miniconda and Create Env

Keeps Python pinned to 3.10 for compatibility with project dependencies.



In [7]:
%%bash
set -e

ROOT="/content/drive/MyDrive/FMPCC"
PERSIST_CONDA="$ROOT/miniconda3"
RUNTIME_CONDA="/content/miniconda3"
CONDA_BIN="$PERSIST_CONDA/bin/conda"
LOCAL_FALLBACK_CONDA="/content/miniconda3_runtime"
CONDA_SNAPSHOT_DIR="$ROOT/cache"
CONDA_SNAPSHOT="$CONDA_SNAPSHOT_DIR/fmpcc_conda_env.tar.gz"
FORCE_REINSTALL_CONDA="${FORCE_REINSTALL_CONDA:-0}"
REFRESH_CONDA_SNAPSHOT="${REFRESH_CONDA_SNAPSHOT:-0}"

# Ensure the runtime path points to the persistent conda directory.
if [ -L "$RUNTIME_CONDA" ]; then
  LINK_TARGET="$(readlink -f "$RUNTIME_CONDA" || true)"
  if [ "$LINK_TARGET" != "$PERSIST_CONDA" ]; then
    rm -f "$RUNTIME_CONDA"
    ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"
  fi
elif [ ! -e "$RUNTIME_CONDA" ]; then
  ln -s "$PERSIST_CONDA" "$RUNTIME_CONDA"
fi

if [ "$FORCE_REINSTALL_CONDA" = "1" ]; then
  echo "FORCE_REINSTALL_CONDA=1 -> reinstalling Miniconda"
  rm -rf "$PERSIST_CONDA"
fi

if [ ! -d "$PERSIST_CONDA" ]; then
  echo "First run detected: no persistent conda directory yet"
fi

# 1) Check if conda exists and works.
NEED_CONDA_INSTALL=0
if [ -x "$CONDA_BIN" ]; then
  chmod +x "$PERSIST_CONDA/bin/python" "$PERSIST_CONDA/bin/conda" || true
  if ! "$CONDA_BIN" --version >/tmp/conda_check.log 2>&1; then
    if grep -qi "Permission denied" /tmp/conda_check.log; then
      echo "Drive conda is not executable (permission denied). Switching to local runtime conda."
      if [ -L "$RUNTIME_CONDA" ] || [ -e "$RUNTIME_CONDA" ]; then
        rm -rf "$RUNTIME_CONDA"
      fi
      ln -s "$LOCAL_FALLBACK_CONDA" "$RUNTIME_CONDA"
      PERSIST_CONDA="$LOCAL_FALLBACK_CONDA"
      CONDA_BIN="$PERSIST_CONDA/bin/conda"
      if [ -x "$CONDA_BIN" ]; then
        echo "Reusing existing local runtime conda."
      else
        echo "No local runtime conda found yet; it will be installed now."
        NEED_CONDA_INSTALL=1
      fi
    else
      echo "Conda exists but failed health check -> reinstalling"
      rm -rf "$PERSIST_CONDA"
      NEED_CONDA_INSTALL=1
    fi
  else
    echo "Conda exists and passed health check -> skip reinstall"
  fi
else
  NEED_CONDA_INSTALL=1
fi

# 2) If there is a problem, reinstall.
if [ "$NEED_CONDA_INSTALL" = "1" ]; then
  if [ -f "$CONDA_SNAPSHOT" ]; then
    echo "Restoring conda snapshot: $CONDA_SNAPSHOT"
    rm -rf "$PERSIST_CONDA"
    mkdir -p "$PERSIST_CONDA"
    if ! tar -xzf "$CONDA_SNAPSHOT" -C "$PERSIST_CONDA" --strip-components=1; then
      echo "Snapshot restore failed -> will run installer fallback"
      rm -rf "$PERSIST_CONDA"
    fi
  fi

  if [ ! -x "$CONDA_BIN" ] || ! "$CONDA_BIN" --version >/dev/null 2>&1; then
    echo "Conda snapshot missing/broken -> installing Miniconda"
    wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
    bash /content/miniconda.sh -b -p "$PERSIST_CONDA" -u
  else
    echo "Conda restored from snapshot"
  fi
fi

"$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
"$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

if ! "$CONDA_BIN" env list | grep -q "^FMPCC "; then
  "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
fi

if [ ! -x "$PERSIST_CONDA/envs/FMPCC/bin/python" ] || [ ! -x "$PERSIST_CONDA/envs/FMPCC/bin/pip" ]; then
  "$CONDA_BIN" remove -n FMPCC --all -y || true
  "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
fi

# If env binaries on Drive are not executable, switch to local runtime conda and rebuild env there.
if ! "$PERSIST_CONDA/envs/FMPCC/bin/python" -V >/tmp/env_python_check.log 2>&1; then
  if grep -qi "Permission denied" /tmp/env_python_check.log; then
    echo "FMPCC env python is not executable on current path. Switching env to local runtime conda."
    if [ -L "$RUNTIME_CONDA" ] || [ -e "$RUNTIME_CONDA" ]; then
      rm -rf "$RUNTIME_CONDA"
    fi
    ln -s "$LOCAL_FALLBACK_CONDA" "$RUNTIME_CONDA"
    PERSIST_CONDA="$LOCAL_FALLBACK_CONDA"
    CONDA_BIN="$PERSIST_CONDA/bin/conda"

    if [ ! -x "$CONDA_BIN" ]; then
      wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/miniconda.sh
      bash /content/miniconda.sh -b -p "$PERSIST_CONDA" -u
    fi

    "$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
    "$CONDA_BIN" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
    "$CONDA_BIN" remove -n FMPCC --all -y || true
    "$CONDA_BIN" create -n FMPCC python=3.10 -y -q
  else
    cat /tmp/env_python_check.log
    exit 1
  fi
fi

"$PERSIST_CONDA/envs/FMPCC/bin/python" -V
"$PERSIST_CONDA/envs/FMPCC/bin/pip" --version

# Save a conda snapshot for fast restore on future Colab runtimes.
mkdir -p "$CONDA_SNAPSHOT_DIR"
if [ ! -f "$CONDA_SNAPSHOT" ] || [ "$REFRESH_CONDA_SNAPSHOT" = "1" ]; then
  echo "Creating conda snapshot: $CONDA_SNAPSHOT"
  if ! tar -czf "$CONDA_SNAPSHOT" -C "$PERSIST_CONDA" .; then
    echo "Snapshot creation failed -> continuing without cache update"
  fi
else
  echo "Conda snapshot exists -> skipping snapshot refresh"
fi



Conda exists and passed health check -> skip reinstall
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
FMPCC env python is not executable on current path. Switching env to local runtime conda.
PREFIX=/content/miniconda3_runtime
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /content/miniconda3_runtime
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channe


EnvironmentLocationNotFound: Not a conda environment: /content/miniconda3_runtime/envs/FMPCC



## 5) Install D3IL (Install Once + Verify)

Uses editable installs for both D3IL core and `gym_avoiding_env`, but skips reinstall when editable links already exist.



In [8]:
%%bash
set -e

PIP="/content/miniconda3/envs/FMPCC/bin/pip"
REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
D3IL="$REPO/d3il"

if [ ! -d "$D3IL/.git" ]; then
  echo "d3il missing/incomplete -> recloning"
  rm -rf "$D3IL"
  git clone https://github.com/ALRhub/d3il.git "$D3IL"
fi

if "$PIP" freeze | grep -Fq "d3il/environments/d3il" && "$PIP" freeze | grep -Fq "d3il/envs/gym_avoiding_env"; then
  echo "D3IL editable installs already present; skipping reinstall"
else
  "$PIP" install -e "$D3IL/environments/d3il"
  "$PIP" install -e "$D3IL/environments/d3il/envs/gym_avoiding_env"
fi

echo "D3IL installed"



Obtaining file:///content/drive/MyDrive/FMPCC/FM-PCC/d3il/environments/d3il
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for environments.d3il.d3il_sim (pyproject.toml): started
  Building editable for environments.d3il.d3il_sim (pyproject.toml): finished with status 'done'
  Created wheel for environments.d3il.d3il_sim: filename=environments_d3il_d3il_sim-0.2-0.editable-py3-none-any.whl size=2970 sha256=1f4c769882c3b278be9fc78934ef0755d9d0c0097c000d934ff2e1c22bae5802
  Stored in directory: /tmp/pip-ephem-wheel-cache-3i6159

## 6) Install Requirements (Install Once + Verify)

Runs validation first and only installs when the environment is missing or inconsistent.



In [9]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
PIP="/content/miniconda3/envs/FMPCC/bin/pip"
PY="/content/miniconda3/envs/FMPCC/bin/python"
PIP_CACHE="/content/drive/MyDrive/FMPCC/.pip-cache"
CACHE_DIR="/content/drive/MyDrive/FMPCC/cache"
REQ_FILE="$REPO/requirements.txt"
REQ_HASH="$(sha256sum "$REQ_FILE" | awk '{print $1}')"
WHEELHOUSE="$CACHE_DIR/wheelhouse_$REQ_HASH"
REQ_STAMP="/content/miniconda3/envs/FMPCC/.fmpcc_requirements_hash"
FORCE_REINSTALL_REQS="${FORCE_REINSTALL_REQS:-0}"

mkdir -p "$PIP_CACHE" "$CACHE_DIR"
cd "$REPO"

if [ "$FORCE_REINSTALL_REQS" = "1" ]; then
  echo "FORCE_REINSTALL_REQS=1 -> forcing requirements reinstall"
  REQ_VALID=0
elif "$PY" - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        importlib.import_module(p)
    except Exception as e:
        ok = False
        print(f'Missing/broken package: {p} ({type(e).__name__}: {e})')

import numpy
if int(numpy.__version__.split('.')[0]) >= 2:
    ok = False
    print(f'Invalid numpy version: {numpy.__version__}; expected 1.x for this workflow')

sys.exit(0 if ok else 2)
PY
then
  REQ_VALID=1
else
  REQ_VALID=0
fi

if [ "$REQ_VALID" = "1" ]; then
  if [ -f "$REQ_STAMP" ] && grep -Fxq "$REQ_HASH" "$REQ_STAMP"; then
    echo "Package validation passed and requirements hash unchanged; skipping reinstall"
  else
    echo "Package validation passed but requirements hash stamp missing/changed; updating stamp only"
    echo "$REQ_HASH" > "$REQ_STAMP"
  fi
else
  echo "Package validation failed; installing requirements"

  # Build or reuse a Drive-backed wheelhouse for faster future reinstalls.
  if [ ! -d "$WHEELHOUSE" ] || [ -z "$(ls -A "$WHEELHOUSE" 2>/dev/null || true)" ]; then
    echo "Building wheelhouse cache at: $WHEELHOUSE"
    mkdir -p "$WHEELHOUSE"
    PIP_CACHE_DIR="$PIP_CACHE" "$PIP" download -r requirements.txt -d "$WHEELHOUSE" || true
  else
    echo "Wheelhouse cache found: $WHEELHOUSE"
  fi

  WHEEL_COUNT="$(find "$WHEELHOUSE" -maxdepth 1 -type f \( -name '*.whl' -o -name '*.tar.gz' -o -name '*.zip' \) | wc -l)"
  if [ "$WHEEL_COUNT" -gt 0 ] && PIP_CACHE_DIR="$PIP_CACHE" "$PIP" install --no-index --find-links "$WHEELHOUSE" -r requirements.txt; then
    echo "Offline wheelhouse install succeeded"
  else
    echo "First run or incomplete wheelhouse -> running online install"
    PIP_CACHE_DIR="$PIP_CACHE" "$PIP" install -r requirements.txt

    # Backfill wheelhouse after a successful online install for faster future runs.
    PIP_CACHE_DIR="$PIP_CACHE" "$PIP" download -r requirements.txt -d "$WHEELHOUSE" || true
  fi

  # pip check may report known non-fatal issues on Colab (platform/extra metadata).
  PIP_CHECK_OUT="$("$PIP" check 2>&1 || true)"
  echo "$PIP_CHECK_OUT"

  UNEXPECTED_PIP_CHECK="$(echo "$PIP_CHECK_OUT" | grep -Ev '(^$|gurobipy .* is not supported on this platform|WARNING: typer .* does not provide the extra .all.)' || true)"
  if [ -n "$UNEXPECTED_PIP_CHECK" ]; then
    echo "Unexpected pip check issues found:"
    echo "$UNEXPECTED_PIP_CHECK"
    exit 1
  else
    echo "Only known non-fatal pip check warnings detected; continuing"
  fi

  echo "$REQ_HASH" > "$REQ_STAMP"
fi

# Quick sanity check
"$PY" - <<'PY'
import numpy, torch
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("python:", __import__("sys").executable)
PY



Missing/broken package: torch (ModuleNotFoundError: No module named 'torch')
Missing/broken package: scipy (ModuleNotFoundError: No module named 'scipy')
Missing/broken package: gymnasium (ModuleNotFoundError: No module named 'gymnasium')
Missing/broken package: gymnasium_robotics (ModuleNotFoundError: No module named 'gymnasium_robotics')
Missing/broken package: minari (ModuleNotFoundError: No module named 'minari')
Missing/broken package: mujoco (ModuleNotFoundError: No module named 'mujoco')
Missing/broken package: diffusers (ModuleNotFoundError: No module named 'diffusers')
Missing/broken package: transformers (ModuleNotFoundError: No module named 'transformers')
Invalid numpy version: 2.2.6; expected 1.x for this workflow
Package validation failed; installing requirements
Wheelhouse cache found: /content/drive/MyDrive/FMPCC/cache/wheelhouse_7f51b2eecd675d29f593a4ec70084ed4251266c758f7a4374628942b33efaa39
Looking in links: /content/drive/MyDrive/FMPCC/cache/wheelhouse_7f51b2eecd675

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 7) Runtime Environment Variables

Includes W&B malformed service cleanup and Colab rendering settings.



In [10]:
import os

FMPCC = '/content/drive/MyDrive/FMPCC/FM-PCC'
D3IL_ROOT = f'{FMPCC}/d3il'
GYM_AV = f'{D3IL_ROOT}/environments/d3il/envs/gym_avoiding_env'

existing_pp = os.environ.get('PYTHONPATH', '')
parts = [FMPCC, D3IL_ROOT, GYM_AV]
if existing_pp:
    parts.append(existing_pp)

os.environ['FMPCC'] = FMPCC
os.environ['PYTHONPATH'] = ':'.join(parts)
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'
os.environ['MPLBACKEND'] = 'agg'

for key in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(key, None)

os.chdir(FMPCC)
print('cwd:', os.getcwd())
print('PYTHONPATH:', os.environ['PYTHONPATH'])



cwd: /content/drive/MyDrive/FMPCC/FM-PCC
PYTHONPATH: /content/drive/MyDrive/FMPCC/FM-PCC:/content/drive/MyDrive/FMPCC/FM-PCC/d3il:/content/drive/MyDrive/FMPCC/FM-PCC/d3il/environments/d3il/envs/gym_avoiding_env:/env/python


## 8) Optional W&B Login



In [11]:
import os
from pathlib import Path
import wandb

KEY_FILE = Path('/content/drive/MyDrive/FMPCC/.wandb_api_key')

if not KEY_FILE.exists():
    raise FileNotFoundError(
        f'Missing W&B key file: {KEY_FILE}. Create it with your API key on one line.'
    )

api_key = KEY_FILE.read_text(encoding='utf-8').strip()
if not api_key:
    raise ValueError(f'W&B key file is empty: {KEY_FILE}')

for k in ('WANDB_SERVICE', 'WANDB__SERVICE'):
    os.environ.pop(k, None)

wandb.finish()
os.environ['WANDB_MODE'] = 'online'
os.environ['WANDB_API_KEY'] = api_key

wandb.login(key=api_key, relogin=True)

print('W&B mode:', os.environ.get('WANDB_MODE'))
print('W&B key file:', KEY_FILE)
print('W&B key loaded:', f'***{api_key[-4:]}' if len(api_key) >= 4 else '***')



/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B mode: online
W&B key file: /content/drive/MyDrive/FMPCC/.wandb_api_key
W&B key loaded: ***KogM


## 9) Full Verification

Validates import chain with the exact env interpreter used for training.



In [12]:
%%bash
set -e

/content/miniconda3/envs/FMPCC/bin/python - <<'PY'
import importlib
import sys

pkgs = [
    'torch', 'numpy', 'scipy', 'gym', 'gymnasium', 'gymnasium_robotics',
    'minari', 'wandb', 'mujoco', 'diffusers', 'transformers'
]

ok = True
for p in pkgs:
    try:
        m = importlib.import_module(p)
        v = getattr(m, '__version__', 'unknown')
        print(f'{p:20s} {v}')
    except Exception as e:
        ok = False
        print(f'{p:20s} NOT IMPORTABLE ({type(e).__name__}: {e})')

import numpy, torch
print('numpy pinned:', numpy.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device:', torch.cuda.get_device_name(0))

major = int(numpy.__version__.split('.')[0])
if major >= 2:
    ok = False
    print('ERROR: numpy 2.x detected, expected 1.26.4 for this workflow')

if not ok:
    sys.exit(2)
PY



torch                2.2.2+cu121
numpy                1.26.4
scipy                1.13.1
gym                  0.26.2
gymnasium            0.29.1
gymnasium_robotics   1.2.4
minari               0.4.3
wandb                0.17.5
mujoco               2.3.7
diffusers            0.31.0
transformers         4.41.2
numpy pinned: 1.26.4
cuda available: True
device: Tesla T4


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


## 10) Dataset Preparation (Avoiding)

### Option A: Use existing zip from old DPCC path

Warning! : It searches the ~15Gb Zip in the DPCC Path, not this FMPCC Path!


This exits quickly if avoiding data already exists.



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
AVOIDING_DATA="$REPO/d3il/environments/dataset/data/avoiding/data"
DATA_ZIP="/content/drive/MyDrive/DPCC/dpcc/d3il/environments/dataset/data/dataset.zip"

if [ -d "$AVOIDING_DATA" ] && [ "$(ls -A "$AVOIDING_DATA")" ]; then
  echo "avoiding data already present: $(ls "$AVOIDING_DATA" | wc -l) files"
  exit 0
fi

if [ ! -f "$DATA_ZIP" ]; then
  echo "dataset zip not found: $DATA_ZIP"
  echo "Skip this cell and use Option B below if needed."
  exit 1
fi

TMP="/content/avoiding_tmp"
rm -rf "$TMP"
mkdir -p "$TMP"
unzip -q "$DATA_ZIP" "avoiding/*" -d "$TMP"
mkdir -p "$REPO/d3il/environments/dataset/data/avoiding"
cp -r "$TMP/avoiding/." "$REPO/d3il/environments/dataset/data/avoiding/"
rm -rf "$TMP"

echo "avoiding dataset ready: $(ls "$AVOIDING_DATA" | wc -l) files"



### Option B: Download full D3IL dataset zip with gdown (only if Option A unavailable)



```
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
DATA_DIR="$REPO/d3il/environments/dataset/data"
ZIP_FILE="$DATA_DIR/dataset.zip"

if [ -f "$ZIP_FILE" ]; then
  echo "zip already exists: $ZIP_FILE"
  exit 0
fi

/content/miniconda3/envs/FMPCC/bin/pip install gdown -q
/content/miniconda3/envs/FMPCC/bin/python -m gdown \
  "https://drive.google.com/uc?id=1SQhbhzV85zf_ltnQ8Cbge2lsSWInxVa8" \
  -O "$ZIP_FILE"

echo "downloaded zip: $ZIP_FILE"



## 11) Smoke Test Train

Short check before full run.



## 12) Full Train

Real-time streaming via `!python`.



In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC

!/content/miniconda3/envs/FMPCC/bin/python FM_test/train_FM.py --seeds 5 --num-seeds 1 --use-wandb --wandb-project FMPCC


In [13]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_test/train_FM_v3.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC

[ train ] Seed source: cli --seeds
[ train ] Training seeds: [6]
[ utils/setup ] Made savepath: logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6
[ train ] Saved seed manifest to: logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/seeds_config.json
wandb: Currently logged in as: llmmail2021 (llmmail2021-technical-university-of-munich). Use `wandb login --relogin` to force relogin
wandb: wandb version 0.25.1 is available!  To upgrade, please run:
wandb:  $ pip install wandb --upgrade
wandb: Tracking run with wandb version 0.17.5
wandb: Run data is saved locally in /content/drive/MyDrive/FMPCC/FM-PCC/wandb/run-20260406_131412-yn84gzv4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run avoiding-d3il-seed-6
wandb: ⭐️ View project at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC
wandb: 🚀 View run at https://wandb.ai/llmmail2021-technical-university-of-munich/FMPCC/runs/yn84gzv4

[utils/config ] Co

KeyboardInterrupt: 

WANDB internel Crushed

## 13) Resume Training (Optional)



In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/train.py --seeds 5 --auto-resume --use-wandb --wandb-project FMPCC
!/content/miniconda3/envs/FMPCC/bin/python FM_test/train_FM.py --seeds 5 --auto-resume --use-wandb --wandb-project FMPCC

In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python FM_hp_tune_test/train_FM_hp_tune.py --seeds 6 --num-seeds 1 --use-wandb --wandb-project FMPCC --auto-resume

## 14) Evaluation and Results



Remember to edit the yaml in /config to choose seeds
and
must write_to_file: True

In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/eval.py
!/content/miniconda3/envs/FMPCC/bin/python FM_test/eval_FM.py

#throw error
#raise Exception("This is a custom error message")


In [14]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_test/eval_FM_v3.py

pybullet build time: Nov 28 2023 23:45:17
[ utils/setup ] Made savepath: logs/avoiding-d3il/plans/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ utils/serialization ] Loading model from logs/avoiding-d3il/flow_matching_v3/H8_K20_Dmodels.diffusion.GaussianDiffusion/6

[ datasets/buffer ] Fields:
    observations: (96, 150, 4)
    actions: (96, 150, 2)
    rewards: (96, 150, 1)
    terminals: (96, 150, 1)
    normed_observations: (96, 150, 4)
    normed_actions: (96, 150, 2)
[ utils/training ] Restored loss history from checkpoint at step 90000

Final IK error (78 iterations):  7.2858622740627195e-06
Final IK error (0 iterations):  7.2858622740627195e-06
------------------------Running avoiding-d3il - top-right-hard - dpcc-r (6)----------------------------
Success rate: 1.0
Constraints satisfied: 0.5
Success rate (goal and constraints): 0.5
Avg number of steps: 82.00 +- 19.00
Avg number of constraint violations: 4.50 +- 4.50
Avg total violation: 0.045 +- 0.045
Average c

### Load Results

If the process crashed change the yaml to resume at crash point.

Save Path logs/avoiding-d3il/plans/H8_K20_Dmodels.GaussianDiffusion/0/results/

In [ ]:
#!/content/miniconda3/envs/FMPCC/bin/python scripts/load_results.py
!/content/miniconda3/envs/FMPCC/bin/python FM_test/load_results_FM.py


In [15]:
!/content/miniconda3/envs/FMPCC/bin/python FM_v3_test/load_results_FM_v3.py

Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
------------------ Variant: dpcc-r ------------------
Success rate (goal): 1.00
Success rate (goal + constraints): 0.67
Success rate (constraints): 0.67
Average steps: 83.17 +- 23.84
Average violations: 13.33 +- 26.00
Average total violations: 0.174 +- 0.351
Average time: 0.33 +- 0.02
$83.2 \pm 23.8$ & $1.00$ & $0.67$ & $13.3 \pm 26.0$ \\
Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
------------------ Variant: dpcc-r-tightened ------------------
Success rate (goal): 1.00
Success rate (goal + constraints): 1.00
Success rate (constraints): 1.00
Average steps: 73.83 +- 19.57
Average violations: 0.00 +- 0.00
Average total violations: 0.000 +- 0.000
Average time: 0.34 +- 0.02
$73.8 \pm 19.6$ & $1.00$ & $1.00$ & $0.0 \pm 0.0$ \\
Eval ODE=10, FlowSteps=10, Beta=(1.5,1.0)
Eval ODE=10, FlowS

## 15) Visualization



In [ ]:
!/content/miniconda3/envs/FMPCC/bin/python scripts/visualize_data_constraints.py



## 16) Archive Logs to Drive



In [ ]:
%%bash
set -e

REPO="/content/drive/MyDrive/FMPCC/FM-PCC"
STAMP="$(date +%Y%m%d_%H%M%S)"
OUT="/content/drive/MyDrive/FMPCC/runs_snapshot/$STAMP"

mkdir -p "$OUT"
if [ -d "$REPO/logs" ]; then
  cp -r "$REPO/logs" "$OUT/"
fi
if [ -d "$REPO/wandb" ]; then
  cp -r "$REPO/wandb" "$OUT/"
fi

echo "snapshot saved: $OUT"


# misc...